In [1]:
class Parser:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def match(self, token):
        if self.pos < len(self.tokens) and self.tokens[self.pos] == token:
            self.pos += 1
            return True
        return False

    # E → T E'
    def E(self):
        pos_backup = self.pos
        if self.T() and self.E_prime():
            return True
        self.pos = pos_backup
        return False

    # E' → + T E' | ε
    def E_prime(self):
        pos_backup = self.pos
        if self.match("+"):
            if self.T() and self.E_prime():
                return True
            self.pos = pos_backup
            return False
        return True  # ε

    # T → F T'
    def T(self):
        pos_backup = self.pos
        if self.F() and self.T_prime():
            return True
        self.pos = pos_backup
        return False

    # T' → * F T' | ε
    def T_prime(self):
        pos_backup = self.pos
        if self.match("*"):
            if self.F() and self.T_prime():
                return True
            self.pos = pos_backup
            return False
        return True  # ε

    # F → (E) | id
    def F(self):
        pos_backup = self.pos
        
        if self.match("("):
            if self.E() and self.match(")"):
                return True
            self.pos = pos_backup
        
        if self.match("id"):
            return True
        
        return False


def analyser(chaine):
    parser = Parser(chaine)
    return parser.E() and parser.pos == len(chaine)

In [ ]:
print(analyser(["id", "+","id", "id"]))  
print(analyser(["id", "*", "id"]))  
print(analyser(["(", "id", "+", "id", ")"]))  
print(analyser(["id", "+"]))  
print(analyser(["id", "+", "id", "*", "id"]))  

False
True
True
False
True


In [10]:
class ASTNode:
    def __init__(self, type_, value=None, children=None):
        self.type = type_
        self.value = value
        self.children = children or []

class ParserAST:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0
        self.id_counter = 0  # pour générer x, y, z...

    def match(self, token):
        if self.pos < len(self.tokens) and self.tokens[self.pos] == token:
            self.pos += 1
            return True
        return False

    # E → T E'
    def E(self):
        t_node = self.T()
        if t_node is None:
            return None
        return self.E_prime(t_node)

    # E' → + T E' | ε
    def E_prime(self, left):
        if self.match("+"):
            t_node = self.T()
            if t_node is None:
                return None
            node = ASTNode('op', '+', [left, t_node])
            return self.E_prime(node)
        return left  # ε

    # T → F T'
    def T(self):
        f_node = self.F()
        if f_node is None:
            return None
        return self.T_prime(f_node)

    # T' → * F T' | ε
    def T_prime(self, left):
        if self.match("*"):
            f_node = self.F()
            if f_node is None:
                return None
            node = ASTNode('op', '*', [left, f_node])
            return self.T_prime(node)
        return left  # ε

    # F → (E) | id
    def F(self):
        if self.match("("):
            e_node = self.E()
            if e_node and self.match(")"):
                return e_node
            return None
        if self.match("id"):
            name = chr(120 + self.id_counter)  # x, y, z
            self.id_counter += 1
            return ASTNode('id', name)
        return None

def analyser_ast(tokens):
    parser = ParserAST(tokens)
    ast = parser.E()
    if ast and parser.pos == len(tokens):
        return ast
    return None

def generer_code(node):
    if node.type == 'id':
        return node.value
    elif node.type == 'op':
        left = generer_code(node.children[0])
        right = generer_code(node.children[1])
        return f"({left} {node.value} {right})"

In [11]:
class ASTNode:
    def __init__(self, type_, value=None, children=None):
        self.type = type_          # 'op' ou 'id'
        self.value = value         # '+', '*', 'id', etc.
        self.children = children or []  # sous-arbres

In [12]:
tokens = ["id", "+", "id", "*", "id"]
parser = ParserAST(tokens)      # ParserAST = version AST
ast = parser.E()
print(generer_code(ast))        # "(x + (y * z))"

(x + (y * z))


In [13]:
def generer_code(node):
    """
    Parcourt l'AST et génère une expression Python valide.
    Gère correctement la priorité pour éviter les parenthèses inutiles.
    """
    if node.type == 'id':
        return node.value

    elif node.type == 'op':
        left = generer_code(node.children[0])
        right = generer_code(node.children[1])
        op = node.value

        # Priorité : * > +
        # Ajouter des parenthèses si le sous-arbre a un opérateur de plus faible priorité
        if node.children[0].type == 'op':
            if precedence(node.children[0].value) < precedence(op):
                left = f"({left})"
        if node.children[1].type == 'op':
            if precedence(node.children[1].value) < precedence(op):
                right = f"({right})"

        return f"{left} {op} {right}"

def precedence(op):
    """Retourne la priorité d'un opérateur (+ ou *)"""
    if op == '*':
        return 2
    elif op == '+':
        return 1
    return 0

In [14]:
tokens1 = ["id", "+", "id"]
tokens2 = ["(", "id", "+", "id", ")", "*", "id"]

ast1 = analyser_ast(tokens1)
ast2 = analyser_ast(tokens2)

print(generer_code(ast1))  # x + y
print(generer_code(ast2))  # (x + y) * z

x + y
(x + y) * z
